# Task 2. Fixture ETL 재실행과 멱등성 검증

## 검증 의도

이 노트북은 외부 RPC 없이 현재 Python 소스 코드가 fixture raw log를 같은 방식으로 처리하는지 확인합니다.
평가자가 확인해야 할 핵심은 `cryptoquant_pipeline.log_normalizer`, ERC-20 decode helper, `delta_writer.write_ethereum_logs_insert_only()`가 같은 natural key 정책을 공유한다는 점입니다.

## 확인하는 흐름

```text
tests/fixtures/rpc_logs.json
→ normalize_logs()
→ ERC-20 Transfer topic/data decode
→ expected_transfers.json 대조
→ malformed payload 분리
→ Delta insert-if-not-exists 2회 실행
→ row count와 duplicate natural key 확인
```

## 검증 경계

- 실제 RPC 호출 없음.
- `data/tmp/src_notebook_fixture_etl` 아래 임시 Delta table만 삭제·재생성함.
- 동일 `chain_id + transaction_hash + log_index`는 1건만 남아야 함.

In [2]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import UTC, datetime
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    """노트북 실행 위치가 달라도 pyproject.toml 기준으로 저장소 루트를 찾음."""
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("pyproject.toml을 찾지 못해 저장소 루트를 확정할 수 없음.")


repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

fixtures_dir = repo_root / "tests" / "fixtures"
print({"repo_root": str(repo_root), "fixtures_dir": str(fixtures_dir)})

{'repo_root': '/workspace', 'fixtures_dir': '/workspace/tests/fixtures'}


## 1. Raw fixture 로드

`rpc_logs.json`은 같은 `transactionHash + logIndex`를 가진 log를 두 번 포함함. 정규화 단계는 payload 변환만 담당하고, 중복 제거는 저장 단계의 natural key 정책으로 확인함.

In [3]:
raw_logs = json.loads((fixtures_dir / "rpc_logs.json").read_text(encoding="utf-8"))

natural_keys = {
    (1, raw_log["transactionHash"].lower(), int(raw_log["logIndex"], 16))
    for raw_log in raw_logs
}
print({"raw_log_count": len(raw_logs), "unique_natural_key_count": len(natural_keys)})
pprint(raw_logs[0])

{'raw_log_count': 2, 'unique_natural_key_count': 1}
{'address': '0xDAC17F958D2EE523A2206206994597C13D831EC7',
 'blockHash': '0xaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa',
 'blockNumber': '0x64',
 'data': '0x0000000000000000000000000000000000000000000000000000000005f5e100',
 'logIndex': '0x1',
 'removed': False,
 'topics': ['0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef',
            '0x0000000000000000000000001111111111111111111111111111111111111111',
            '0x0000000000000000000000005754284f345afc66a98fbb0a0afe71e0f007b949'],
 'transactionHash': '0xbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb',
 'transactionIndex': '0x0'}


## 2. Raw log 정규화

정규화 결과는 Delta raw schema에 맞춘 row임. `block_timestamp_utc`, `interval_start_utc`, `interval_end_utc`, `ingested_at_utc`는 재실행과 감사 추적에 필요한 시간 경계임.

In [4]:
from cryptoquant_pipeline.log_normalizer import normalize_logs


interval_start = datetime(2024, 1, 1, 0, 0, tzinfo=UTC)
interval_end = datetime(2024, 1, 1, 1, 0, tzinfo=UTC)
ingested_at = datetime(2024, 1, 1, 1, 5, tzinfo=UTC)

# fixture의 blockNumber=0x64이므로 100번 블록 timestamp만 주입함.
normalized = normalize_logs(
    raw_logs,
    block_timestamps_utc={100: datetime(2024, 1, 1, 0, 5, tzinfo=UTC)},
    chain_id=1,
    interval_start_utc=interval_start,
    interval_end_utc=interval_end,
    ingested_at_utc=ingested_at,
)

print(
    {
        "normalized_log_count": len(normalized.rows),
        "invalid_log_count": normalized.invalid_log_count,
    }
)
pprint(normalized.rows[0])

{'normalized_log_count': 2, 'invalid_log_count': 0}
{'block_date_utc': datetime.date(2024, 1, 1),
 'block_hash': '0xaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa',
 'block_number': 100,
 'block_timestamp_utc': datetime.datetime(2024, 1, 1, 0, 5, tzinfo=datetime.timezone.utc),
 'chain_id': 1,
 'contract_address': '0xdac17f958d2ee523a2206206994597c13d831ec7',
 'data_raw': '0x0000000000000000000000000000000000000000000000000000000005f5e100',
 'data_uint256_decimal_text': '100000000',
 'data_uint256_decode_status': 'DECIMAL38_AVAILABLE',
 'ingested_at_utc': datetime.datetime(2024, 1, 1, 1, 5, tzinfo=datetime.timezone.utc),
 'interval_end_utc': datetime.datetime(2024, 1, 1, 1, 0, tzinfo=datetime.timezone.utc),
 'interval_start_utc': datetime.datetime(2024, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'log_index': 1,
 'removed': False,
 'topic0': '0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef',
 'topic1': '0x000000000000000000000000111111111111111111111

## 3. ERC-20 Transfer 디코딩

float 변환 없이 `Decimal`로 raw uint256 값을 다룸. 주소는 indexed topic의 마지막 20 bytes에서 복원함.

In [5]:
from cryptoquant_pipeline.log_normalizer import (
    amount_with_decimals,
    decode_uint256_decimal,
    is_erc20_transfer_topic,
    topic_to_address,
)


decoded_by_key = {}
for row in normalized.rows:
    if not is_erc20_transfer_topic(row["topic0"]):
        continue

    key = (row["chain_id"], row["transaction_hash"], row["log_index"])
    raw_amount = decode_uint256_decimal(str(row["data_raw"]))
    decoded_by_key.setdefault(
        key,
        {
            "token_address": row["contract_address"],
            "from_address": topic_to_address(row["topic1"]),
            "to_address": topic_to_address(row["topic2"]),
            "amount_raw_decimal": str(raw_amount),
            "amount_usdt": str(amount_with_decimals(raw_amount, 6)),
        },
    )

decoded_transfers = list(decoded_by_key.values())
print({"decoded_transfer_count_after_key_dedupe": len(decoded_transfers)})
pprint(decoded_transfers)

{'decoded_transfer_count_after_key_dedupe': 1}
[{'amount_raw_decimal': '100000000',
  'amount_usdt': '100',
  'from_address': '0x1111111111111111111111111111111111111111',
  'to_address': '0x5754284f345afc66a98fbb0a0afe71e0f007b949',
  'token_address': '0xdac17f958d2ee523a2206206994597c13d831ec7'}]


## 4. 예상 Transfer 값과 대조

`expected_transfers.json`은 fixture 기준 기대값임. 이 셀은 실험 결과가 테스트 fixture의 의미와 일치하는지만 확인함.

In [6]:
expected_transfers = json.loads(
    (fixtures_dir / "expected_transfers.json").read_text(encoding="utf-8")
)
expected = expected_transfers[0]
actual = decoded_transfers[0]

assert actual["token_address"] == expected["token_address"]
assert actual["from_address"] == expected["from_address"]
assert actual["to_address"] == expected["to_address"]
assert actual["amount_raw_decimal"] == expected["amount_raw_decimal"]

print("fixture transfer expectation matched")

fixture transfer expectation matched


## 5. Malformed payload 분리 실험

잘못된 hex quantity는 정규화 row로 만들지 않고 `invalid_log_count`로 집계함. 재시도해도 해결되지 않는 schema/payload 오류는 네트워크 오류와 분리해야 함.

In [7]:
malformed_raw_logs = [*raw_logs, {"blockNumber": "not-hex"}]
malformed_result = normalize_logs(
    malformed_raw_logs,
    block_timestamps_utc={100: datetime(2024, 1, 1, 0, 5, tzinfo=UTC)},
    chain_id=1,
    interval_start_utc=interval_start,
    interval_end_utc=interval_end,
    ingested_at_utc=ingested_at,
)

assert len(malformed_result.rows) == len(normalized.rows)
assert malformed_result.invalid_log_count == 1
print(
    {
        "input_count": len(malformed_raw_logs),
        "normalized_log_count": len(malformed_result.rows),
        "invalid_log_count": malformed_result.invalid_log_count,
    }
)

{'input_count': 3, 'normalized_log_count': 2, 'invalid_log_count': 1}


## 6. Delta insert-only 재실행 검증

`chain_id + transaction_hash + log_index`를 natural key로 보고, 같은 batch를 다시 쓰면 row 수가 늘지 않아야 함. 이 셀은 `data/tmp/src_notebook_fixture_etl` 아래 임시 Delta table을 지웠다가 다시 생성함.

In [8]:
from cryptoquant_pipeline.delta_writer import (
    count_duplicate_natural_keys,
    count_rows,
    write_ethereum_logs_insert_only,
)


tmp_root = (repo_root / "data" / "tmp").resolve()
table_root = tmp_root / "src_notebook_fixture_etl"
table_path = table_root / "ethereum_logs"
if tmp_root not in table_root.resolve().parents:
    raise RuntimeError("Delta 실험 삭제 경로가 data/tmp 밖으로 벗어남.")
if table_root.exists():
    shutil.rmtree(table_root)

first_write = write_ethereum_logs_insert_only(normalized.rows, table_path=table_path)
second_write = write_ethereum_logs_insert_only(normalized.rows, table_path=table_path)
summary = {
    "first_inserted_row_count": first_write.inserted_row_count,
    "first_duplicate_skipped_count": first_write.duplicate_skipped_count,
    "second_inserted_row_count": second_write.inserted_row_count,
    "second_duplicate_skipped_count": second_write.duplicate_skipped_count,
    "row_count_after_second_write": count_rows(table_path),
    "duplicate_natural_key_count": count_duplicate_natural_keys(table_path),
    "table_path": str(table_path),
}

assert summary["first_inserted_row_count"] == 1
assert summary["second_inserted_row_count"] == 0
assert summary["row_count_after_second_write"] == 1
assert summary["duplicate_natural_key_count"] == 0
pprint(summary)

{'duplicate_natural_key_count': 0,
 'first_duplicate_skipped_count': 1,
 'first_inserted_row_count': 1,
 'row_count_after_second_write': 1,
 'second_duplicate_skipped_count': 2,
 'second_inserted_row_count': 0,
 'table_path': '/workspace/data/tmp/src_notebook_fixture_etl/ethereum_logs'}
